# ME672 Assignment 3: Timber roof frame, plane-stress FE model (Generalized)
Units: m, N, Pa throughout.

In [1]:
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from mpi4py import MPI

import gmsh
import pyvista
from pyvista.core.errors import PyVistaDeprecationWarning

import dolfinx
from dolfinx import default_scalar_type, mesh
from dolfinx.io import gmsh as gmshio
from dolfinx.fem import (Constant, Expression, Function, functionspace,
                         locate_dofs_topological, dirichletbc, form, assemble_scalar)
from dolfinx.fem.petsc import LinearProblem
from dolfinx.plot import vtk_mesh
from ufl import (TestFunction, TrialFunction, Measure, FacetNormal,
                 inner, dot, grad, sym, tr, Identity, as_vector)

from fem_utils import (plot_sparseness)

warnings.filterwarnings("ignore", category=PyVistaDeprecationWarning, module="pyvista")

## Parameters

In [2]:
b = 0.076            # out-of-plane thickness, m
s = 3.0              # roof width carried by one frame, m
support_len = 0.100  # length of left and right supports, m

E, nu = 10e9, 0.30
rho, g = 770.0, 9.81 
f_t, f_c, f_s = 5e6, 5e6, 1e6

h = 0.03             
element_order = 2
brep_file = "Timber_Frame_Rebuilt-Frame_Solid.brep"

## Mesh (gmsh → DOLFINx)

In [3]:
gmsh.initialize()
gmsh.option.setNumber("General.Terminal", 0)
gmsh.model.add("frame")
occ = gmsh.model.occ

occ.importShapes(brep_file)
occ.dilate(occ.getEntities(), 0, 0, 0, 1e-3, 1e-3, 1e-3)    # mm -> m
occ.synchronize()

# Auto-detect bounding box to make it generic
xmin, ymin, zmin, xmax, ymax, zmax = gmsh.model.getBoundingBox(-1, -1)
x_mid = (xmin + xmax) / 2.0

p1 = occ.addPoint(xmin + support_len, ymin, 0)
p2 = occ.addPoint(xmax - support_len, ymin, 0)

# Apply fragment to ALL surfaces, not just the first one
surfs = gmsh.model.getEntities(2)
occ.fragment(surfs, [(0, p1), (0, p2)])
occ.synchronize()

h_sup = 0.005            

sup_pts = []
support_coords = [(xmin, ymin), (xmin + support_len, ymin), 
                  (xmax - support_len, ymin), (xmax, ymin)]

for (x0, y0) in support_coords:
    found = gmsh.model.getEntitiesInBoundingBox(x0-1e-6, y0-1e-6, -1, x0+1e-6, y0+1e-6, 1, dim=0)
    sup_pts += [t for _, t in found]

fd = gmsh.model.mesh.field.add("Distance")
gmsh.model.mesh.field.setNumbers(fd, "PointsList", sup_pts)
ft = gmsh.model.mesh.field.add("Threshold")
gmsh.model.mesh.field.setNumber(ft, "InField", fd)
gmsh.model.mesh.field.setNumber(ft, "SizeMin", h_sup)
gmsh.model.mesh.field.setNumber(ft, "SizeMax", h)
gmsh.model.mesh.field.setNumber(ft, "DistMin", 0.02)
gmsh.model.mesh.field.setNumber(ft, "DistMax", 0.30)
gmsh.model.mesh.field.setAsBackgroundMesh(ft)
gmsh.option.setNumber("Mesh.MeshSizeFromPoints", 0)
gmsh.option.setNumber("Mesh.MeshSizeExtendFromBoundary", 0)

# Add ALL surfaces to the physical group
s_tags = [tag for dim, tag in gmsh.model.getEntities(2)]
gmsh.model.addPhysicalGroup(2, s_tags, 1)

gmsh.option.setNumber("Mesh.MeshSizeMax", h)
gmsh.model.mesh.generate(2)
gmsh.model.mesh.setOrder(element_order)

mesh_data = gmshio.model_to_mesh(gmsh.model, MPI.COMM_WORLD, 0, gdim=2)
domain = mesh_data.mesh
gmsh.finalize()

tdim = domain.topology.dim
fdim = tdim - 1

## Boundary tags

In [4]:
domain.topology.create_connectivity(fdim, tdim)

# Exact mesh coordinates from diagnostic
mn_x, mx_x = -0.050, 8.050
mn_y, mx_y = -0.143, 1.476
support_len = 0.100
x_mid = (mn_x + mx_x) / 2.0

# TOPOLOGICAL LOCATOR FOR ROOF (To apply surface loads)
def roof_facets(x):
    # Grabs exterior faces above the bottom chord
    return x[1] > mn_y + 0.20

f_roof = mesh.locate_entities_boundary(domain, fdim, roof_facets)
mid = mesh.compute_midpoints(domain, fdim, f_roof)
f_roofL, f_roofR = f_roof[mid[:, 0] < x_mid], f_roof[mid[:, 0] >= x_mid]

indices = np.hstack([f_roofL, f_roofR]).astype(np.int32)
values  = np.hstack([np.full(len(f_roofL), 21), np.full(len(f_roofR), 22)]).astype(np.int32)
order = np.argsort(indices)
facet_tags = mesh.meshtags(domain, fdim, indices[order], values[order])

dx_ = Measure("dx", domain=domain)
ds_ = Measure("ds", domain=domain, subdomain_data=facet_tags)
n = FacetNormal(domain)

print(f"Roof facets loaded: {len(f_roof)}")

Roof facets loaded: 0


## Function space and plane-stress constitutive law

In [5]:
V = functionspace(domain, ("Lagrange", element_order, (2,)))

mu = E/(2*(1+nu))
lmbda_ps = E*nu/(1-nu**2)

def epsilon(u): return sym(grad(u))
def sigma(u): return lmbda_ps*tr(epsilon(u))*Identity(2) + 2*mu*epsilon(u)

u = TrialFunction(V)
v = TestFunction(V)

## Supports, Loads and weak form

In [6]:
# 1. GEOMETRICAL LOCATORS FOR SUPPORTS (To pin the frame)
def left_support_nodes(x):
    return (x[1] <= mn_y + 0.05) & (x[0] <= mn_x + support_len + 0.02)

def right_support_nodes(x):
    return (x[1] <= mn_y + 0.05) & (x[0] >= mx_x - support_len - 0.02)

dofs_L = dolfinx.fem.locate_dofs_geometrical(V, left_support_nodes)

# Collapse the subspace to allow geometrical location in DOLFINx
V_y, _ = V.sub(1).collapse()
dofs_R_2d = dolfinx.fem.locate_dofs_geometrical((V.sub(1), V_y), right_support_nodes)

# Extract just the 1D array of original DOFs for the boundary condition
dofs_R = dofs_R_2d[:, 0]

print(f"Left support nodes pinned: {len(dofs_L)}")
print(f"Right support nodes pinned: {len(dofs_R)}")

bc_L = dirichletbc(np.zeros(2, dtype=default_scalar_type), dofs_L, V)
bc_R = dirichletbc(default_scalar_type(0.0), dofs_R, V.sub(1))
bcs = [bc_L, bc_R]

f_v = Constant(domain, default_scalar_type((0.0, -rho*g)))
pL = Constant(domain, default_scalar_type(0.0))
pR = Constant(domain, default_scalar_type(0.0))
q_L = as_vector((0.0, -pL*s*abs(n[1])))
q_R = as_vector((0.0, -pR*s*abs(n[1])))
a = b*inner(sigma(u), epsilon(v))*dx_
L = b*dot(f_v, v)*dx_ + dot(q_L, v)*ds_(21) + dot(q_R, v)*ds_(22)

cases = { 
    "G":   (1.10, 1.10),
    "A":   (1.10, 0.35),
    "A_m": (0.35, 1.10),
    "U":   (-1.15, -1.15),
    "W":   (-1.15, 0.85),
    "W_m": (0.85, -1.15),
}

problem = LinearProblem(a, L, bcs=bcs, petsc_options_prefix="frame_",
                        petsc_options={"ksp_type": "preonly", "pc_type": "lu"})

TypeError: list indices must be integers or slices, not tuple

## Equilibrium, Stresses, and Execution

In [ ]:
# Verify that supports were actually found
if len(dofs_L) == 0 or len(dofs_R) == 0:
    raise ValueError(f"Supports not found! Check geometry bounding box: mn_x={mn_x}, mn_y={mn_y}, mx_x={mx_x}")

# Pass the arrays directly without indexing
nodes_L = dofs_L
nodes_R = dofs_R

one = Constant(domain, default_scalar_type(1.0))
I = lambda expr: domain.comm.allreduce(assemble_scalar(form(expr)), op=MPI.SUM)
area = I(one*dx_)
proj_L = I(abs(n[1])*ds_(21))
proj_R = I(abs(n[1])*ds_(22))

def reaction(nodes, comp):
    w = Function(V)
    # Safely assign 1.0 to the correct component of the specific nodes
    w.x.array[2 * np.array(nodes) + comp] = 1.0
    work = b*inner(sigma(uh), epsilon(w))*dx_ - (b*dot(f_v, w)*dx_ + dot(q_L, w)*ds_(21) + dot(q_R, w)*ds_(22))
    return I(work)

W0 = functionspace(domain, ("DG", 1))
X  = W0.element.interpolation_points
xy = W0.tabulate_dof_coordinates()[:, :2]

ends = np.array(support_coords)
dist = np.min(np.linalg.norm(xy[:, None, :] - ends[None, :, :], axis=2), axis=1)
excluded = dist < 0.05

def stress_component(uh, i, j):
    fh = Function(W0)
    fh.interpolate(Expression(sigma(uh)[i, j], X))
    return fh.x.array.copy()

def check_case(uh):
    sxx, syy, sxy = stress_component(uh, 0, 0), stress_component(uh, 1, 1), stress_component(uh, 0, 1)
    c, R = (sxx + syy) / 2, np.sqrt(((sxx - syy) / 2)**2 + sxy**2)
    s1, s2 = c + R, c - R
    zero = np.zeros_like(s1)
    smax, smin = np.maximum.reduce([zero, s1, s2]), np.minimum.reduce([zero, s1, s2])
    
    tens, comp, shear = smax / f_t, -smin / f_c, (smax - smin) / 2 / f_s
    keep = ~excluded
    ux, uy = uh.x.array[0::2], uh.x.array[1::2]
    
    return {
        "max_ux_mm": 1e3*np.abs(ux).max(), "max_uy_mm": 1e3*np.abs(uy).max(),
        "tension": tens[keep].max(), "compression": comp[keep].max(), "shear": shear[keep].max()
    }

results = []
for case, (p_l, p_r) in cases.items():
    pL.value, pR.value = p_l * 1e3, p_r * 1e3
    uh = problem.solve()
    
    W_tot = rho * g * b * area + (pL.value * proj_L + pR.value * proj_R) * s
    Rx_L, Ry_L = reaction(nodes_L, 0), reaction(nodes_L, 1)
    Rx_R, Ry_R = reaction(nodes_R, 0), reaction(nodes_R, 1)
    
    row = check_case(uh)
    row.update({
        "case": case, "load_kN": W_tot / 1e3,
        "Rx_L_N": Rx_L, "Ry_L_kN": Ry_L / 1e3,
        "Rx_R_N": Rx_R, "Ry_R_kN": Ry_R / 1e3
    })
    results.append(row)

df_results = pd.DataFrame(results).set_index("case")
df_results["feasible"] = ((df_results[["tension", "compression", "shear"]] <= 1.0).all(axis=1) 
                          & (df_results["max_ux_mm"] <= 33.3) & (df_results["max_uy_mm"] <= 33.3))
df_results.to_csv("results_summary.csv")
print(df_results.round(3))

      max_ux_mm  max_uy_mm  tension  compression  shear  load_kN  Rx_L_N  \
case                                                                       
G           NaN        NaN      NaN          NaN    NaN    5.496     NaN   
A           NaN        NaN      NaN          NaN    NaN    5.496     NaN   
A_m         NaN        NaN      NaN          NaN    NaN    5.496     NaN   
U           NaN        NaN      NaN          NaN    NaN    5.496     NaN   
W           NaN        NaN      NaN          NaN    NaN    5.496     NaN   
W_m         NaN        NaN      NaN          NaN    NaN    5.496     NaN   

      Ry_L_kN  Rx_R_N  Ry_R_kN  feasible  
case                                      
G         NaN     NaN      NaN     False  
A         NaN     NaN      NaN     False  
A_m       NaN     NaN      NaN     False  
U         NaN     NaN      NaN     False  
W         NaN     NaN      NaN     False  
W_m       NaN     NaN      NaN     False  
